# Reserve 1: Saros-Grid-SSM - Google Colab Runner
**Project:** Solar/Wind Spatio-Temporal SSM for Spanish REE Electricity Grid  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Can continuous-time selective state space recurrence coupled with graph spatial diffusion predict 15-minute nodal renewable injections with wMAPE < 6.5% and curtailment F1 >= 0.75 without $O(N^2)$ attention?

Configured for Google Colab (Free T4 or Pro A100/L4) and local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Unassigning Colab GPU immediately to prevent idle burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB")
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 2: Dependencies & Dual-Mode Standalone Engine
try:
    from saros_grid_ssm.model import GridSpatioTemporalSSM
    from saros_grid_ssm.ree_data import generate_synthetic_ree_grid, REESpanishGridDataset
    from saros_grid_ssm.metrics import compute_wmape, compute_curtailment_f1, compute_grid_metrics
    print("Loaded Saros-Grid-SSM from package.")
except ImportError:
    print("Package not on path; loading self-contained standalone implementation (zero-network)...")
    import math
    from typing import Dict, List, Tuple, Union
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset

    def build_spanish_grid_adjacency(num_nodes: int = 50) -> np.ndarray:
        adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        corridors = [
            (0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 18), (1, 14), (2, 18), (3, 31), (3, 28),
            (4, 5), (4, 7), (4, 8), (5, 9), (6, 9), (6, 10), (6, 11), (7, 10), (10, 11),
            (4, 13), (5, 15), (9, 15), (11, 49), (12, 13), (12, 14), (13, 15), (12, 27),
            (14, 15), (14, 17), (15, 16), (16, 17), (17, 18), (16, 46), (16, 49), (17, 21), (18, 29),
            (19, 20), (19, 21), (19, 22), (20, 22), (21, 22), (19, 39), (19, 44), (20, 40), (22, 44),
            (23, 24), (23, 25), (23, 26), (23, 27), (23, 28), (23, 30), (23, 31),
            (24, 26), (24, 29), (24, 38), (25, 26), (25, 30), (25, 33), (27, 30), (28, 31), (29, 40),
            (32, 33), (32, 35), (33, 34), (34, 35), (32, 41), (33, 25), (34, 30),
            (36, 37), (36, 38), (36, 41), (37, 38), (37, 39), (38, 39), (39, 40), (41, 25),
            (42, 43), (42, 44), (42, 45), (43, 44), (44, 48), (44, 19),
            (46, 47), (46, 48), (47, 49), (48, 44), (49, 11)
        ]
        for u, v in corridors:
            if u < num_nodes and v < num_nodes:
                adj[u, v] = 1.0
                adj[v, u] = 1.0
        return adj

    def normalize_adjacency(adj: np.ndarray) -> np.ndarray:
        adj_loop = adj + np.eye(adj.shape[0], dtype=np.float32)
        degrees = np.sum(adj_loop, axis=1)
        d_inv_sqrt = np.power(degrees, -0.5)
        d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
        return np.diag(d_inv_sqrt) @ adj_loop @ np.diag(d_inv_sqrt)

    def generate_synthetic_ree_grid(num_days: int = 14, num_nodes: int = 50, seed: int = 2026):
        rng = np.random.RandomState(seed)
        timesteps_per_day = 96
        total = num_days * timesteps_per_day
        solar_cap = np.zeros(num_nodes, dtype=np.float32)
        wind_cap = np.zeros(num_nodes, dtype=np.float32)
        base_demand = np.zeros(num_nodes, dtype=np.float32)
        export_limit = np.zeros(num_nodes, dtype=np.float32)

        for i in range(num_nodes):
            if i < 4:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 100.0, 50.0, 3200.0, 3800.0
            elif 4 <= i <= 13:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 2600.0, 600.0, 900.0, 1600.0
            elif 14 <= i <= 18:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 1800.0, 1200.0, 600.0, 1800.0
            elif 19 <= i <= 22:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 600.0, 3000.0, 800.0, 2000.0
            elif 23 <= i <= 31:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 800.0, 2200.0, 700.0, 2000.0
            elif 32 <= i <= 35:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 250.0, 2300.0, 900.0, 1700.0
            elif 42 <= i <= 45:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 500.0, 700.0, 2800.0, 3200.0
            else:
                solar_cap[i], wind_cap[i], base_demand[i], export_limit[i] = 700.0, 700.0, 1200.0, 1800.0

        t_steps = np.arange(total)
        hour = ((t_steps % timesteps_per_day) * 15.0) / 60.0
        sol_diurnal = np.power(np.maximum(0.0, np.sin(np.pi * np.clip((hour - 7.0) / 14.0, 0, 1))), 1.3)
        wnd_diurnal = 0.55 + 0.25 * np.cos(2 * np.pi * (hour - 3.0) / 24.0) + 0.15 * np.sin(4 * np.pi * (hour - 15.0) / 24.0)
        dem_diurnal = 0.60 + 0.25 * np.exp(-0.5 * ((hour - 11.5) / 2.5) ** 2) + 0.30 * np.exp(-0.5 * ((hour - 21.0) / 2.2) ** 2)

        inputs = np.zeros((total, num_nodes, 4), dtype=np.float32)
        injections = np.zeros((total, num_nodes, 2), dtype=np.float32)
        curtailments = np.zeros((total, num_nodes, 1), dtype=np.float32)
        phase = rng.uniform(0, 2 * np.pi, size=(num_nodes,))

        for t in range(total):
            d_idx = t // timesteps_per_day
            c_factor = 0.85 + 0.15 * np.sin(2 * np.pi * (d_idx / 7.0) + phase)
            w_factor = 0.90 + 0.25 * np.cos(2 * np.pi * (d_idx / 5.0) + phase)
            s_pot = np.maximum(0.0, solar_cap * sol_diurnal[t] * c_factor + rng.normal(0, 5, size=(num_nodes,)))
            w_pot = np.maximum(0.0, wind_cap * wnd_diurnal[t] * w_factor + rng.normal(0, 10, size=(num_nodes,)))
            dem = base_demand * dem_diurnal[t] * (1.0 + rng.normal(0, 0.02, size=(num_nodes,)))

            inputs[t, :, 0], inputs[t, :, 1], inputs[t, :, 2], inputs[t, :, 3] = s_pot, w_pot, dem, export_limit
            net_excess = s_pot + w_pot - dem - export_limit
            is_curt = net_excess > 0.0
            curtailments[t, :, 0] = is_curt.astype(np.float32)
            injections[t, :, 0] = np.maximum(0.0, np.where(is_curt, s_pot - 0.6 * net_excess, s_pot))
            injections[t, :, 1] = np.maximum(0.0, np.where(is_curt, w_pot - 0.4 * net_excess, w_pot))

        adj = build_spanish_grid_adjacency(num_nodes)
        return {
            "inputs": inputs, "injections": injections, "curtailments": curtailments,
            "adjacency": adj, "adjacency_norm": normalize_adjacency(adj)
        }

    class REESpanishGridDataset(Dataset):
        def __init__(self, inputs, injections, curtailments, window_size=96, stride=96):
            self.inputs = torch.from_numpy(inputs).float()
            self.injections = torch.from_numpy(injections).float()
            self.curtailments = torch.from_numpy(curtailments).float()
            self.window_size = window_size
            self.indices = list(range(0, inputs.shape[0] - window_size + 1, stride))
        def __len__(self): return len(self.indices)
        def __getitem__(self, idx):
            s = self.indices[idx]
            e = s + self.window_size
            return self.inputs[s:e], self.injections[s:e], self.curtailments[s:e]

    class GraphSpatialDiffusion(nn.Module):
        def __init__(self, d_in: int, d_out: int, num_hops: int = 2):
            super().__init__()
            self.num_hops = num_hops
            self.weights = nn.ParameterList([nn.Parameter(torch.empty(d_in, d_out)) for _ in range(num_hops + 1)])
            self.bias = nn.Parameter(torch.zeros(d_out))
            self.norm = nn.LayerNorm(d_out)
            for w in self.weights: nn.init.xavier_uniform_(w)
        def forward(self, x, adj_norm):
            p = x
            out = torch.matmul(p, self.weights[0])
            for k in range(1, self.num_hops + 1):
                p = torch.einsum("nm,btmd->btnd", adj_norm, p)
                out = out + torch.matmul(p, self.weights[k])
            return self.norm(F.gelu(out + self.bias))

    class SelectiveSSMBlock(nn.Module):
        def __init__(self, d_model: int, d_state: int = 16, bidirectional: bool = True, dropout: float = 0.05):
            super().__init__()
            self.d_model, self.d_state, self.bidirectional = d_model, d_state, bidirectional
            self.log_alpha = nn.Parameter(torch.linspace(0.1, 2.0, d_state).unsqueeze(0))
            self.proj_delta = nn.Linear(d_model, d_state)
            self.proj_b = nn.Linear(d_model, d_state)
            self.proj_c = nn.Linear(d_model, d_state)
            self.d_skip = nn.Parameter(torch.ones(d_model))
            self.out_proj = nn.Linear(d_model, d_model)
            self.norm = nn.LayerNorm(d_model)
        def forward(self, x):
            b, t, n, d = x.shape
            x_flat = x.permute(0, 2, 1, 3).reshape(b * n, t, d)
            delta = F.softplus(self.proj_delta(x_flat))
            b_bar = delta * self.proj_b(x_flat)
            c_t = self.proj_c(x_flat)
            a_bar = torch.exp(delta * (-torch.exp(self.log_alpha)).unsqueeze(0))

            h = torch.zeros(b * n, self.d_state, device=x.device, dtype=x.dtype)
            y_steps = []
            for s in range(t):
                x_mean = x_flat[:, s, :].mean(dim=-1, keepdim=True)
                h = a_bar[:, s, :] * h + b_bar[:, s, :] * x_mean
                y_steps.append(h * c_t[:, s, :])
            y_stacked = torch.stack(y_steps, dim=1)
            if self.d_state != self.d_model:
                rep = math.ceil(self.d_model / self.d_state)
                y_stacked = y_stacked.repeat(1, 1, rep)[:, :, :self.d_model]
            out = self.norm(self.out_proj(y_stacked) + x_flat * self.d_skip)
            return out.reshape(b, n, t, d).permute(0, 2, 1, 3)

    class GridForecastingHead(nn.Module):
        def __init__(self, d_model: int, out_features: int = 2):
            super().__init__()
            self.inj_head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, out_features), nn.Softplus())
            self.curt_head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Linear(d_model // 2, 1), nn.Sigmoid())
        def forward(self, h): return self.inj_head(h), self.curt_head(h)

    class GridSpatioTemporalSSM(nn.Module):
        def __init__(self, in_features=4, out_features=2, d_model=64, d_state=16, num_spatial_hops=2, num_layers=2, bidirectional=True, dropout=0.05):
            super().__init__()
            self.in_proj = nn.Linear(in_features, d_model)
            self.spatial = nn.ModuleList([GraphSpatialDiffusion(d_model, d_model, num_hops=num_spatial_hops) for _ in range(num_layers)])
            self.temporal = nn.ModuleList([SelectiveSSMBlock(d_model, d_state=d_state, bidirectional=bidirectional, dropout=dropout) for _ in range(num_layers)])
            self.head = GridForecastingHead(d_model, out_features=out_features)
        def forward(self, x, adj_norm):
            h = self.in_proj(x)
            for s, t in zip(self.spatial, self.temporal):
                h = h + s(h, adj_norm)
                h = h + t(h)
            inj, curt = self.head(h)
            return {"injection_mw": inj, "curtailment_prob": curt}

    def compute_wmape(y_true, y_pred, eps=1e-5):
        diff = torch.abs(y_true - y_pred).sum()
        denom = torch.abs(y_true).sum() + eps
        return (diff / denom).item()

    def compute_curtailment_f1(y_true, y_pred_prob, threshold=0.5, eps=1e-7):
        yt = (y_true > 0.5).detach().cpu().numpy().astype(int).flatten()
        yp = (y_pred_prob >= threshold).detach().cpu().numpy().astype(int).flatten()
        tp = np.sum((yt == 1) & (yp == 1))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        prec = tp / (tp + fp + eps)
        rec = tp / (tp + fn + eps)
        f1 = 2 * (prec * rec) / (prec + rec + eps)
        return {"curtailment_precision": float(prec), "curtailment_recall": float(rec), "curtailment_f1": float(f1)}

    def compute_grid_metrics(injection_true, injection_pred, curtailment_true, curtailment_pred_prob):
        wmape = compute_wmape(injection_true, injection_pred)
        diff = injection_true - injection_pred
        mae = torch.mean(torch.abs(diff)).item()
        rmse = torch.sqrt(torch.mean(diff ** 2)).item()
        res = compute_curtailment_f1(curtailment_true, curtailment_pred_prob)
        res.update({"wmape": wmape, "mae": mae, "rmse": rmse})
        return res

print("[Standby Check] All Saros-Grid-SSM components ready for zero-network execution!")




In [ ]:
# Step 3: Generate REE 50-Node Grid & Initialize Model
seed = 2026
torch.manual_seed(seed)
num_nodes = 50

raw_data = generate_synthetic_ree_grid(num_days=21, num_nodes=num_nodes, seed=seed)
adj_norm = torch.from_numpy(raw_data["adjacency_norm"]).float().to(device)

train_split = 14 * 96
train_dataset = REESpanishGridDataset(
    inputs=raw_data["inputs"][:train_split],
    injections=raw_data["injections"][:train_split],
    curtailments=raw_data["curtailments"][:train_split],
    window_size=96,
    stride=48
)
val_dataset = REESpanishGridDataset(
    inputs=raw_data["inputs"][train_split:],
    injections=raw_data["injections"][train_split:],
    curtailments=raw_data["curtailments"][train_split:],
    window_size=96,
    stride=96
)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

model = GridSpatioTemporalSSM(
    in_features=4,
    out_features=2,
    d_model=64,
    d_state=16,
    num_spatial_hops=2,
    num_layers=2,
    bidirectional=True,
    dropout=0.05
).to(device)

print(f"Initialized GridSpatioTemporalSSM ({sum(p.numel() for p in model.parameters()):,} params)")


In [ ]:
# Step 4: Run Smoke Training Loop (15-min dispatch)
import torch.nn as nn

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
huber_loss_fn = nn.SmoothL1Loss(beta=10.0)
bce_loss_fn = nn.BCELoss()

model.train()
steps, max_steps = 0, 100
t0 = time.time()

while steps < max_steps:
    for x_b, y_inj_b, y_curt_b in train_loader:
        x_b, y_inj_b, y_curt_b = x_b.to(device), y_inj_b.to(device), y_curt_b.to(device)
        optimizer.zero_grad()
        preds = model(x_b, adj_norm)
        loss = huber_loss_fn(preds["injection_mw"], y_inj_b) + 5.0 * bce_loss_fn(preds["curtailment_prob"], y_curt_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        steps += 1
        if steps % 10 == 0:
            wmape_step = compute_wmape(y_inj_b, preds["injection_mw"])
            print(f"Step {steps:02d}/{max_steps} | Loss: {loss.item():.4f} | Batch wMAPE: {wmape_step * 100:.2f}%")
        if steps >= max_steps:
            break

print(f"Training elapsed: {time.time() - t0:.2f}s")


In [ ]:
# Step 5: Evaluate Pre-Registered Gate Criteria
model.eval()
with torch.no_grad():
    for x_val, y_inj_val, y_curt_val in val_loader:
        x_val, y_inj_val, y_curt_val = x_val.to(device), y_inj_val.to(device), y_curt_val.to(device)
        preds_val = model(x_val, adj_norm)
        metrics = compute_grid_metrics(
            y_inj_val, preds_val["injection_mw"],
            y_curt_val, preds_val["curtailment_prob"]
        )

wmape_pct = metrics["wmape"] * 100.0
gate_passed = metrics["wmape"] < 0.065 and metrics["curtailment_f1"] >= 0.75

print("=" * 60)
print(f"Day-Ahead Injection wMAPE : {wmape_pct:.2f}% (Target: < 6.50%)")
print(f"Curtailment F1 Score      : {metrics['curtailment_f1']:.4f} (Target: >= 0.750)")
print(f"Pre-Registered Gate 1     : {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 5: Credit Protection & Automatic Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
